# B2 数据清洗、划分与数据泄漏

**竞赛模块：B · 机器学习算法应用** · **建议学习约 20 分钟**

预处理不是随意清洗：填补缺失值的中位数和标准化均值必须由训练集学得，不能偷偷读取测试集信息。

> 零基础学习顺序：先读术语和说明 → 运行示例 → 理解代码 → 完成练习 → 概念自测。按 Shift+Enter 运行单元格。

## 一、先认识术语

- **缺失值 NaN**：无法获得的数据，numpy 中常写 np.nan
- **拟合 fit**：根据当前数据计算预处理参数/模型参数
- **数据泄漏**：使用了预测时不应该知道的信息，导致评估失真

训练前要处理缺失值、数值缩放和类别编码。通常先划分训练/测试，再只在训练集拟合预处理器，防止泄漏。

## 二、运行示例

逐格执行并查看输出。这里的代码使用本地/合成数据，遇到缺失输入会有提示。不要直接跳过基础代码。

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
X = np.array([[1.0],[2],[np.nan],[4],[5],[6],[7],[8],[9],[10]])
y = [0,0,0,0,0,1,1,1,1,1]
Xtr, Xte, ytr, yte = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
imp = SimpleImputer(strategy="median")
scaler = StandardScaler()
Xtr = scaler.fit_transform(imp.fit_transform(Xtr))
Xte = scaler.transform(imp.transform(Xte))
print(Xtr.shape, Xte.shape)

## 三、逐句解释

1. train_test_split 先随机划分 8:2 并根据 y 分层。

2. SimpleImputer 的 fit_transform 在训练集上求中位数并填补。

3. StandardScaler 再在训练集计算均值和标准差；测试集只能 transform。

### 额外提醒

- test_size=0.2 表示留 20% 测试集
- imputer 根据训练数据计算填补值
- 测试数据只能 transform

## 四、重点函数与参数

### train_test_split

- **写法**：`train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`
- **参数**：test_size：测试占比；random_state：固定随机；stratify：分层依据
- **返回**：返回 X_train,X_test,y_train,y_test
- **特别注意**：分类标签分层有助于稳定训练/测试类别比例

### StandardScaler

- **写法**：`scaler.fit_transform(X_train) / scaler.transform(X_test)`
- **参数**：fit_transform：学习后转换；transform：仅转换
- **返回**：返回标准化数组
- **特别注意**：测试集不能重新拟合

## 五、运行结果核验

训练/测试 shape 应分别为 (8,1) 和 (2,1)，执行过程中不会出现 NaN。

> 结果可能因随机种子、包版本或操作系统略有差异。重点核对形状、字段类型、输出文件和运行是否报错。

## 六、动手实践（先自己写）

**一级：入门**：修改缺失值填补策略为 mean 并比较。

**二级：巩固**：尝试把 test_size 改为 0.3，打印两组标签分布。

**三级：挑战**：将 Imputer/Scaler/LogisticRegression 组合成 Pipeline。

**本课小挑战**：将 StandardScaler 改成使用训练数据拟合后变换全部测试数据，确认代码没有 test.fit。

建议在下面的单元格中独立填写代码，先从一级入门开始，再做二级、三级。

In [ ]:
# TODO 1：完成一级入门练习

# 在这里输入代码


In [ ]:
# TODO 2：完成二级巩固练习

# 在这里输入代码


In [ ]:
# TODO 3：完成三级挑战练习

# 在这里输入代码


## 七、独立练习自动检查

使用已 fit 的 imp、scaler 处理 [[3.0]]，结果赋给 answer_b2。

请在上面的 TODO 练习单元或新建单元自行编写代码，生成变量 `answer_b2`。下面的检查单元**不会自动完成练习**。若未通过，请根据错误提示修改代码；通过后会在项目根目录的 `_beginner_grades/` 保存本地检查记录。


In [ ]:
# TODO 4：检查本课练习（请先自己生成 answer_b2）
from pathlib import Path as _Path
import sys as _sys
for _candidate in [_Path.cwd(), *_Path.cwd().parents]:
    if (_candidate / "beginner_checkpoints.py").is_file():
        _sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("未找到 beginner_checkpoints.py，请从项目根目录启动 Jupyter")
from beginner_checkpoints import check_lesson
check_lesson("B2", globals())


## 八、常见错误和排查

1. 对全部 X 先 fit_transform 再 split，会把测试集统计信息泄漏进训练流程。

2. 在已标准化的 Xtr 上再次调用 imp.fit_transform，虽然通常不报错但已经不是原始数据处理逻辑。

**完成标准**：能解释为什么训练集是 fit_transform、测试集只能 transform。

## 九、参考写法（先完成练习再展开）

<details><summary>展开查看原练习参考代码</summary>

```python
train_scaled = scaler.fit_transform(imp.fit_transform(Xtr))
test_scaled = scaler.transform(imp.transform(Xte))
```

参考写法不一定覆盖全部三级拓展练习；应以实际运行结果为准。

</details>

## 十、概念自测

**问题**：哪种写法会造成数据泄漏？

A. 训练集 fit，测试集 transform

B. 训练和测试集都分别 fit_transform

C. 只对训练集 fit

<details><summary>展开答案与解析</summary>

正确选项：**B**。不能用测试集学习预处理参数。

</details>

## 十、衔接竞赛实操

学完本模块基础课后，返回学习平台，进入 **6.2.2** 竞赛强化题。

[返回零基础课程网页](../../beginner_course.html#B2)

[打开 6.2.2 竞赛实操题](../../practice.html?question=6.2.2)